# Évaluation finale du classifier sur le test

Évaluer le modèle retenu sur tout le jeu de test, avec son vectoriseur, ses SVM
et ses huit seuils sauvegardés. Les résultats constituent donc l’évaluation indépendante après sélection sur validation.

## 1. Charger le modèle et le test


In [1]:
import hashlib
import html
import json
import unicodedata
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss, precision_recall_fscore_support,
)
from threadpoolctl import threadpool_limits

racine = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "classifier" / "resultats" / "modele.joblib").is_file()), None)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

chemin_modele = racine / "classifier" / "resultats" / "modele.joblib"
empreinte_modele = hashlib.sha256(chemin_modele.read_bytes()).hexdigest()
modele = joblib.load(chemin_modele)
contenu_test = (racine / "data" / "test.jsonl").read_bytes()
empreinte_test = hashlib.sha256(contenu_test).hexdigest()
test = pd.DataFrame([json.loads(ligne) for ligne in
                    contenu_test.decode("utf-8").splitlines() if ligne.strip()])

tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
assert [tag.replace("_", " ") for tag in modele["tags"]] == tags_cibles
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
assert modele["colonnes_cibles"] == colonnes_cibles
assert len(test) > 0 and test[colonnes_cibles].isin([0, 1]).all().all()
assert test["fichier_source"].is_unique
assert np.shape(modele["seuils"]) == (8,) and np.isfinite(modele["seuils"]).all()
y_test = test[colonnes_cibles].to_numpy(dtype=np.int64)
exercices_test = test.to_dict("records")
print(f"Classifier chargé pour {len(test)} exercices de test et {len(tags_cibles)} catégories à prédire.")

Classifier chargé pour 732 exercices de test et 8 catégories à prédire.


## 2. Prédictions par exercice

In [ ]:
def predire_exercice(exercice, modele):
    sections = []
    for champ, titre in modele["champs_texte"].items():
        valeur = exercice.get(champ)
        if isinstance(valeur, str):
            contenu = unicodedata.normalize("NFC", html.unescape(valeur)).strip()
            if contenu:
                sections.append(f"{titre}\n{contenu}")
    texte = "\n\n".join(sections)
    if not texte.strip():
        raise ValueError("L’énoncé doit contenir au moins une section textuelle non vide.")
    matrice = modele["vectoriseur"].transform([texte])
    scores = modele["classifieur"].decision_function(matrice)[0]
    assert np.shape(scores) == (8,) and np.isfinite(scores).all()
    return (scores >= np.asarray(modele["seuils"])).astype(np.int64)


predictions_test, durees_ms = [], []
with threadpool_limits(limits=1):
    for exercice in exercices_test[:3]:
        predire_exercice(exercice, modele)
    for exercice in exercices_test:
        debut = perf_counter()
        prediction = predire_exercice(exercice, modele)
        durees_ms.append(1000 * (perf_counter() - debut))
        predictions_test.append(prediction)
predictions_test = np.stack(predictions_test)
durees_ms = np.asarray(durees_ms)
assert predictions_test.shape == y_test.shape and len(durees_ms) == len(test)


def afficher_tags(ligne):
    return ", ".join(tag for tag, actif in zip(tags_cibles, ligne) if actif) or "Aucun tag cible"


resultats_exercices = pd.DataFrame({
    "Fichier": test["fichier_source"].to_numpy(),
    "Tags attendus": [afficher_tags(ligne) for ligne in y_test],
    "Tags prédits": [afficher_tags(ligne) for ligne in predictions_test],
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(resultats_exercices)


,Fichier,Tags attendus,Tags prédits
0,sample_100.json,math,"math, number theory"
1,sample_1000.json,Aucun tag cible,Aucun tag cible
2,sample_1005.json,Aucun tag cible,Aucun tag cible
3,sample_1006.json,strings,Aucun tag cible
4,sample_1022.json,strings,strings
5,sample_1028.json,Aucun tag cible,Aucun tag cible
6,sample_1031.json,math,math
7,sample_1042.json,"math, probabilities","math, probabilities"
8,sample_1046.json,geometry,"graphs, geometry"
9,sample_1051.json,Aucun tag cible,Aucun tag cible


## 3. Résultats globaux

In [3]:
metriques_globales = pd.DataFrame([{
    "F1 macro": f1_score(y_test, predictions_test, average="macro", zero_division=0),
    "F1 micro": f1_score(y_test, predictions_test, average="micro", zero_division=0),
    "Exact Match Ratio": accuracy_score(y_test, predictions_test),
    "Hamming Loss": hamming_loss(y_test, predictions_test),
    "Inférence médiane (ms/exercice)": float(np.median(durees_ms)),
}])
with pd.option_context("display.max_columns", None):
    display(metriques_globales.round(4))

,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,0.6672,0.6146,0.5601,0.0675,0.5477


## 4. Résultats par tag

In [4]:
precisions, rappels, f1_tags, supports = precision_recall_fscore_support(
    y_test, predictions_test, average=None, zero_division=0)
metriques_par_tag = pd.DataFrame({
    "Tag": tags_cibles, "Support": supports.astype(int),
    "Précision": precisions, "Rappel": rappels, "F1": f1_tags,
})
display(metriques_par_tag.round(4))

,Tag,Support,Précision,Rappel,F1
0,math,211,0.5647,0.6209,0.5914
1,graphs,81,0.5857,0.5062,0.5430
2,strings,63,0.6282,0.7778,0.6950
3,number theory,53,0.5000,0.3962,0.4421
4,trees,49,0.8049,0.6735,0.7333
5,geometry,25,0.7083,0.6800,0.6939
6,games,16,0.8667,0.8125,0.8387
7,probabilities,14,0.9091,0.7143,0.8000
